# 02 · Row-level security: a store manager only sees their own store (executed)

`store_row_filter` (a Unity Catalog SQL function) is attached with `WITH ROW FILTER … ON (store_id)` to
10 store-keyed silver and gold tables in the pipeline. The `store_access` table maps principals to stores:
HQ and the app/pipeline identities get `*`, the Koramangala manager gets `STR-001`.

Below, the **same queries** run as HQ and as the Koramangala store manager (a real service principal
logging in with OAuth), first in SQL, then in **Genie**.

In [1]:
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
print(df(f"SELECT role, store_id FROM {FQ}.store_access ORDER BY store_id, role").to_string(index=False))
print()
for line in df(f"DESCRIBE FUNCTION EXTENDED {FQ}.store_row_filter").iloc[:, 0]:
    if line.strip().split(":")[0] in ("Function", "Input", "Returns", "Comment", "Body"):
        print(" ".join(line.split()))
rf = df(f"""SELECT table_name FROM {CATALOG}.information_schema.row_filters
           WHERE table_schema = '{SCHEMA}' ORDER BY table_name""")
print(f"\ntables with the row filter attached ({len(rf)}):", ", ".join(rf.table_name))

                   role store_id
app:kaapi-bricks-finale        *
      app:kaapi-ops-mcp        *
                   eval        *
                     hq        *
          store_manager  STR-001



Function: fevm_cme_conde_catalog.kaapi_bricks.store_row_filter
Input: row_store_id STRING
Returns: BOOLEAN
Comment: Row filter: TRUE when the current principal is HQ (*) or mapped to this store.
Body: EXISTS ( SELECT 1 FROM store_access a WHERE a.principal = current_user() AND (a.store_id = '*' OR a.store_id = row_store_id) )



tables with the row filter attached (10): gold_delivery_exceptions, gold_inventory_position, gold_open_purchase_orders, gold_product_demand, gold_store_daily_kpis, gold_waste_summary, inventory_transactions, orders, promotion_redemptions, purchase_orders


In [3]:
mgr = store_manager_client()
Q = f"""SELECT 'orders' AS table_name, COUNT(DISTINCT store_id) AS stores, COUNT(*) AS rows FROM {FQ}.orders
UNION ALL SELECT 'gold_inventory_position', COUNT(DISTINCT store_id), COUNT(*) FROM {FQ}.gold_inventory_position
UNION ALL SELECT 'gold_open_purchase_orders', COUNT(DISTINCT store_id), COUNT(*) FROM {FQ}.gold_open_purchase_orders
UNION ALL SELECT 'gold_store_daily_kpis', COUNT(DISTINCT store_id), COUNT(*) FROM {FQ}.gold_store_daily_kpis"""
print("As HQ:"); print(sql_as(hq, Q).to_string(index=False))
print("\nAs the Koramangala store manager (STR-001):"); print(sql_as(mgr, Q).to_string(index=False))

As HQ:


               table_name stores   rows
                   orders     36 200000
  gold_inventory_position     37    814
gold_open_purchase_orders     37    147
    gold_store_daily_kpis     36   3693

As the Koramangala store manager (STR-001):


               table_name stores rows
                   orders      1 6606
  gold_inventory_position      1   22
gold_open_purchase_orders      1    8
    gold_store_daily_kpis      1  103


## The same question in Genie, asked by each identity

In [4]:
q = "How many stores have sales data, and what was total revenue per store in April 2026? Show the top 5 stores."
for who, client in (("HQ", hq), ("Koramangala store manager", mgr)):
    r = genie_ask(q, client)
    print(f"=== {who}  ({r['status']}, {r['seconds']} s) ===")
    print(r["answer"][:1200]); print()

=== HQ  (completed, 14.4 s) ===
## Store Revenue — April 2026

A total of **36 stores** have sales data recorded overall. For April 2026 specifically, here are the top 5 stores by total revenue:

**Top 5 stores by revenue April 2026 and total store count with sales**

| store_id | total_revenue | total_stores |
| --- | --- | --- |
| STR-003 | 344262.75 | 36 |
| STR-011 | 342765.0 | 36 |
| STR-002 | 336926.75 | 36 |
| STR-021 | 324260.25 | 36 |
| STR-001 | 323557.5 | 36 |

Store **STR-003** led with $344,262.75 in revenue, followed closely by STR-011 ($342,765.00) and STR-002 ($336,926.75).



=== Koramangala store manager  (completed, 15.8 s) ===
Only one store appears in this dataset.

## Only One Store Has Sales Data

The dataset contains sales data for just **1 store** (STR-001), so there isn't a top-5 to show — the single store's total revenue in April 2026 was **$323,557.50**.

**Revenue per store, April 2026**

| store_id | total_revenue |
| --- | --- |
| STR-001 | 323557.5 |

